# Modeling Template

This is a template made for loading data from thousands of random Pokemon generation 9 battles from https://pokemonshowdown.com/ and using features from a csv to create models which predict the winner. 

# Outline
# Parsing Pokemon Swaps
## Section 1: Modeling
### 1.1 Logistic Regression
    1.1.1 Baseline model
    1.1.2 Improvements
### 1.2 Decision Tree
### 1.3 Random Forest
### 1.4 Histogram-based Gradient Boosting Classification Tree
### 1.5 XGBoost
## Section 2: Model Comparisons
### 2.1 AUC score
### 2.2 Cross Validation Score
## Section 3: Best Model Analysis
### 3.1 ROC Curve
### 3.2 Confusion Matrix

In [1]:
# Import statements
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_curve
from sklearn.metrics import ConfusionMatrixDisplay
from xgboost import XGBClassifier

import sys
from pathlib import Path
import zipfile
import tempfile

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "summer26-pokemon-battle-predictor" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
sys.path.append(str(PROJECT_ROOT / "tools"))
from battle import Battle

In [2]:
# Load in training dataframe
path_to_train = "../data/data_cleaned.csv.zip"
train_df = pd.read_csv(path_to_train)
train_df = train_df[train_df["p1elo0"] > 0]
print(train_df.columns.tolist())

# Load in testing dataframe
path_to_test = "../data/test_data_cleaned.csv.zip"
test_df = pd.read_csv(path_to_test)
test_df = test_df[test_df["p1elo0"] > 0]
print(test_df.columns.tolist())

['format', 'id', 'p1_win', 'ratedQ', 'n_turns', 'start_time', 'end_time', 'duration', 'p1name', 'p1side', 'p1elo0', 'p1elo1', 'p2name', 'p2side', 'p2elo0', 'p2elo1', 'type_diversity_diff', 'num_boosting_abilities_diff', 'num_move_boosters_diff', 'total_stat_diff', 'p1_total_adv', 'p1_revealed_team_size', 'p2_revealed_team_size', 'M11_name', 'M11_speciesId', 'M11_used', 'M11_gender', 'M11_shinyQ', 'M11_level', 'M11_ability', 'M11_item', 'M11_teraType', 'M11_role', 'M11_mv1', 'M11_mv2', 'M11_mv3', 'M11_mv4', 'M11_type1', 'M11_type2', 'M11_hp', 'M11_atk', 'M11_def', 'M11_spa', 'M11_spd', 'M11_spe', 'M11_off', 'M12_name', 'M12_speciesId', 'M12_used', 'M12_gender', 'M12_shinyQ', 'M12_level', 'M12_ability', 'M12_item', 'M12_teraType', 'M12_role', 'M12_mv1', 'M12_mv2', 'M12_mv3', 'M12_mv4', 'M12_type1', 'M12_type2', 'M12_hp', 'M12_atk', 'M12_def', 'M12_spa', 'M12_spd', 'M12_spe', 'M12_off', 'M13_name', 'M13_speciesId', 'M13_used', 'M13_gender', 'M13_shinyQ', 'M13_level', 'M13_ability', 'M13_i

# Parsing Pokemon Swaps

In [3]:
# Function for getting stats about player switches
def get_switch_stats(battle):
    '''
    Gets following stats from Pokemon showdown logs:

    - <player>_switches
        Total number of player switches excluding lead Pokemon and fainting
    - <player>_switched_early
        Flags whether a player swapped their lead Pokemon on turn 1 before making a move
    '''

    # Ignore the first "switch" that is actually just the lead pokemon
    p1_switches = -1
    p2_switches = -1

    p1_seen_move = False
    p2_seen_move = False

    p1_switched_early = 0
    p2_switched_early = 0

    # Flag that says whether the pokemon has fainted and needs replacement
    p1_needs_replacement = False
    p2_needs_replacement = False

    turns = 0

    for line in battle.log.split("\n"):
        #print(line)
        # Trying to ignore switches caused by fainting
        if line.startswith("|faint|"):
            if "|p1" in line:
                p1_needs_replacement = True

            elif "|p2" in line:
                p2_needs_replacement = True

        # increment nummber of turns
        #if line.startswith("|turn|"):
        #    turns = int(line.split("|")[2])

        fields = line.split("|")

        if len(fields) < 3:
            continue

        if line.startswith("|move|"):
            #print(line)

            slot = fields[2]

            if slot.startswith("p1"):
                p1_seen_move = True

            elif slot.startswith("p2"):
                p2_seen_move = True
                
        elif line.startswith("|switch|"):

            slot = fields[2]

            if slot.startswith("p1"):

                if not p1_needs_replacement:
                    p1_switches += 1

                p1_needs_replacement = False

                if not p1_seen_move and p1_switches > 0:
                    p1_switched_early = 1

            elif slot.startswith("p2"):

                if not p2_needs_replacement:
                    p2_switches += 1

                p2_needs_replacement = False

                if not p2_seen_move and p2_switches > 0:
                    p2_switched_early = 1
                    
    return {
        #"turns": turns,
        "p1_switches": p1_switches,
        "p2_switches": p2_switches,
        "p1_switched_early": p1_switched_early,
        "p2_switched_early": p2_switched_early
    }

# Load in individual replays
replay_root = DATA_DIR / "replays"

replay_zips = [
    replay_root / "gen9randombattles_1.zip",
    replay_root / "gen9randombattles_2.zip",
    replay_root / "gen9randombattles_3.zip",
]

'''
files = []
for replay_dir in replay_dirs:
    files.extend(sorted(replay_dir.glob("*.json")))

print(f"There are {len(files)} replay files")
#print(replay_dir)
#print(replay_dir.exists())
#print(len(files))
#print(files)

rows = []

for file in files:
    try:
        battle = Battle(str(file))
    #except Exception as e:
        #print(f"\nFailed on {file}")
        #print(type(e).__name__, e)
        #break   
    except Exception:
        continue
    stats = get_switch_stats(battle)
    rows.append({"id": battle.id, **stats})
'''
rows = []

for zip_path in replay_zips:

    print(f"Reading {zip_path.name}")

    with zipfile.ZipFile(zip_path) as z:

        for name in z.namelist():

            if not name.endswith(".json"):
                continue

            # Extract the replay to a temporary file
            with tempfile.TemporaryDirectory() as tmpdir:

                extracted = Path(z.extract(name, path=tmpdir))

                try:
                    battle = Battle(str(extracted))
                except Exception:
                    continue

                stats = get_switch_stats(battle)

                rows.append({
                    "id": battle.id,
                    **stats
                })

switch_df = pd.DataFrame(rows)
# Create dataframe with the output of get_switch_stats
switch_df = pd.DataFrame(rows)

print(switch_df.columns)
#print(switch_df.head())

# remove string at the beginning of the battle id
switch_df["id"] = (
    switch_df["id"]
    .str.replace("gen9randombattle-", "", regex=False)
    .astype(int)
)

# Ignore unranked matches
#ranked_df = train_df[train_df["p1elo0"] > 0]
#print(len(train_df))
#print(len(ranked_df))

# Merge main dataframe with switched stats dataframe
train_df = train_df.merge(switch_df, on="id", how="left")
print(len(train_df))

Reading gen9randombattles_1.zip
Reading gen9randombattles_2.zip
Reading gen9randombattles_3.zip
Index(['id', 'p1_switches', 'p2_switches', 'p1_switched_early',
       'p2_switched_early'],
      dtype='object')
12381


In [4]:
# Swapping rates for test data
zip_path = replay_root / "gen9randombattles_testdata.zip"

#test_files = []
#test_files.extend(sorted(test_dir.glob("*.json")))

test_rows = []

with zipfile.ZipFile(zip_path) as z:

    for name in z.namelist():

        if not name.endswith(".json"):
            continue

        # Extract temporarily
        with tempfile.TemporaryDirectory() as tmpdir:

            extracted = Path(z.extract(name, path=tmpdir))

            battle = Battle(str(extracted))

            stats = get_switch_stats(battle)

            test_rows.append({
                "id": battle.id,
                **stats
            })
'''
for test_file in test_files:
    try:
        test_battle = Battle(str(test_file))  
    except Exception as e:
        print(f"\nFailed on {test_file}")
        print(type(e).__name__, e)
        break
    #except Exception:
     #   continue
    test_stats = get_switch_stats(test_battle)
    test_rows.append({"id": test_battle.id, **test_stats})
'''
# Create dataframe with the output of get_switch_stats
test_switch_df = pd.DataFrame(test_rows)

print(test_switch_df.columns)
print(test_switch_df.head())

#print(test_dir)
#print(test_dir.exists())
#print(len(test_files))

# remove string at the beginning of the battle id
test_switch_df["id"] = (
    test_switch_df["id"]
    .str.replace("gen9randombattle-", "", regex=False)
    .astype(int)
)

# Ignore unranked matches
#ranked_df = train_df[train_df["p1elo0"] > 0]
#print(len(train_df))
#print(len(ranked_df))

# Merge main dataframe with switched stats dataframe
test_df = test_df.merge(test_switch_df, on="id", how="left")
print(len(test_df))


Index(['id', 'p1_switches', 'p2_switches', 'p1_switched_early',
       'p2_switched_early'],
      dtype='object')
                            id  p1_switches  p2_switches  p1_switched_early  \
0  gen9randombattle-2644695683            4            3                  1   
1  gen9randombattle-2645572246            1            4                  1   
2  gen9randombattle-2644684138            3            4                  0   
3  gen9randombattle-2645499496            6            8                  1   
4  gen9randombattle-2645496299            8           10                  1   

   p2_switched_early  
0                  0  
1                  0  
2                  0  
3                  0  
4                  1  
4217


# Section 1: Modeling
## 1.1 Logistic Regression
### 1.1.1 Baseline Model

We begin with a simple model using only one feature to predict battle outcome. Following models will add features to see if this baseline model can be improved.

In [5]:
########## BASELINE MODEL ##########
base_feature = ["p1elo0"]
X_train_base = train_df[base_feature]
y_train = train_df["p1_win"]

X_test_base = test_df[base_feature]
y_test = test_df["p1_win"]

baseline = LogisticRegression(
    max_iter=1000
)

baseline.fit(X_train_base, y_train)

base_pred = baseline.predict_proba(X_test_base)[:,1]

### 1.1.2 Improvements

In [6]:
# get speed difference as a feature
p1_speed_cols = [f"M1{i}_spe" for i in range(1, 7)]
p2_speed_cols = [f"M2{i}_spe" for i in range(1, 7)]

# Training data
train_df["p1_total_speed"] = train_df[p1_speed_cols].sum(axis=1)
train_df["p2_total_speed"] = train_df[p2_speed_cols].sum(axis=1)
train_df["speed_diff"] = (
    train_df["p1_total_speed"] - train_df["p2_total_speed"]
)

# Test data
test_df["p1_total_speed"] = test_df[p1_speed_cols].sum(axis=1)
test_df["p2_total_speed"] = test_df[p2_speed_cols].sum(axis=1)
test_df["speed_diff"] = (
    test_df["p1_total_speed"] - test_df["p2_total_speed"]
)

In [7]:
########## IMPROVED MODEL ##########
features = [
    "p1elo0",
    #"total_stat_diff",
    "p1_switched_early",
    #"speed_diff",
    #"type_diversity_diff"
]
X_train = train_df[features]

X_test = test_df[features]

lr = LogisticRegression(
    max_iter=1000
)

lr.fit(X_train, y_train)

lr_pred = lr.predict_proba(X_test)[:,1]

## 1.2 Decision Tree

In [8]:
# Do a grid search for decision tree parameters
dt_param_grid = {
    "max_depth": [3, 5, 8, 10, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=0),
    param_grid=dt_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

print("Best parameters:")
print(dt_grid.best_params_)

Best parameters:
{'max_depth': 8, 'min_samples_leaf': 1, 'min_samples_split': 5}


In [9]:
dt = DecisionTreeClassifier(
    random_state=0,
    max_depth = 8,
    min_samples_leaf= 1,
    min_samples_split= 5
)

dt.fit(X_train, y_train)

dt_pred = dt.predict_proba(X_test)[:,1]

## 1.3 Random Forest

In [10]:
# Do a grid search for random forest parameters
rf_param_grid = {
    "max_depth": [4, 6, 8, 10],
    "min_samples_leaf": [5, 10, 20],
    "n_estimators": [200, 500, 800]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=0),
    param_grid=rf_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

Best parameters:
{'max_depth': 4, 'min_samples_leaf': 5, 'n_estimators': 200}


In [11]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=4,
    min_samples_leaf=5,
    random_state=0,
)

rf.fit(X_train, y_train)

rf_pred = rf.predict_proba(X_test)[:,1]

## 1.4 Histogram-based Gradient Boosting Classification Tree

In [12]:
# Do a grid search for HistGradientBoosting parameters
hgb_param_grid = {
    "max_depth": [4, 6, 8, 10],
    "min_samples_leaf": [5, 10, 20],
    "max_iter": [100, 200, 500] 
}

grid = GridSearchCV(
    HistGradientBoostingClassifier(random_state=0),
    param_grid=hgb_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

Best parameters:
{'max_depth': 4, 'max_iter': 100, 'min_samples_leaf': 5}


In [13]:
hgb = HistGradientBoostingClassifier(
    max_iter=100,
    max_depth=4,
    min_samples_leaf=5,
    random_state=0,
)

hgb.fit(X_train, y_train)

hgb_pred = hgb.predict_proba(X_test)[:, 1]

## 1.5 XGBoost

In [ ]:
# Do a grid search for XGBoost parameters
xgb_param_grid = {
    "max_depth": [4, 6, 8, 10],
    "min_child_weight": [5, 10, 20], 
    "n_estimators": [200, 500, 800] 
}

grid = GridSearchCV(
    XGBClassifier(random_state=0),
    param_grid=xgb_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    min_child_weight=10,
    random_state=0,
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict_proba(X_test)[:, 1]

# Section 2: Model Comparisons
## 2.1 AUC score

In [ ]:
print(f"AUC Scores")
print(f"Baseline Model: {roc_auc_score(y_test, base_pred)}")
print(f"Linear Regression: {roc_auc_score(y_test, lr_pred)}")
print(f"Decision Tree: {roc_auc_score(y_test, dt_pred)}")
print(f"Random Forest: {roc_auc_score(y_test, rf_pred)}")
print(f"HistGradientBoosting: {roc_auc_score(y_test, hgb_pred)}")
print(f"XGBoost: {roc_auc_score(y_test, xgb_pred)}")

## 2.2 Cross Validation Score

In [ ]:
base_cvscore = cross_val_score(baseline, X_train, y_train, cv=5, scoring="roc_auc")
lr_cvscore = cross_val_score(lr, X_train, y_train, cv=5, scoring="roc_auc")
dt_cvscore = cross_val_score(dt, X_train, y_train, cv=5, scoring="roc_auc")
rf_cvscore = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc")
hgb_cvscore = cross_val_score(hgb, X_train, y_train, cv=5, scoring="roc_auc")
xgb_cvscore = cross_val_score(xgb, X_train, y_train, cv=5, scoring="roc_auc")

print("Cross-Validation Scores") 
print(f"Baseline Linear Regression: {base_cvscore.mean():.3f} +/- {lr_cvscore.std():.3f}")
print(f"Linear Regression: {lr_cvscore.mean():.3f} +/- {lr_cvscore.std():.3f}")
print(f"Decision Tree: {dt_cvscore.mean():.3f} +/- {dt_cvscore.std():.3f}")
print(f"Random Forest: {rf_cvscore.mean():.3f} +/- {rf_cvscore.std():.3f}")
print(f"HistGradientBoosting: {hgb_cvscore.mean():.3f} +/- {hgb_cvscore.std():.3f}")
print(f"XGBoost: {xgb_cvscore.mean():.3f} +/- {xgb_cvscore.std():.3f}")

# Section 3: Best Model Analysis
## 3.1 ROC Curve

#### Choose best model and uncomment it below
 - Linear regression model = lr
 - Decision tree model = dt
 - Random forest model = rf

In [ ]:
# Choose best model
#best_model = lr
best_model = dt
#best_model = rf
#best_model = hgb
#best_model = xgb

In [ ]:
prob = best_model.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, prob)

plt.figure(figsize=(6,6))

plt.plot(
    fpr,
    tpr,
    label=f"Best Model (AUC={roc_auc_score(y_test, prob):.3f})"
)

plt.plot([0,1],[0,1],"k--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.show()

## 3.2 Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test,
    y_test
)